In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [4]:
class DeepSeekMOE(nn.Module):
    """
    Simple MoE block:
      - num_shared experts are always applied and summed.
      - num_experts gated experts are selected via top-k router.
      - Inputs are assumed to be [batch_size, h_model].
    """

    def __init__(
        self,
        h_model: int,
        num_experts: int,
        num_shared: int,
        top_k: int,
        hidden_dim: int,
    ) -> None:
        super().__init__()

        assert top_k <= num_experts, "top_k must be <= num_experts"

        self.h_model = h_model
        self.num_experts = num_experts
        self.num_shared = num_shared
        self.top_k = top_k

        # Shared experts (always applied, then summed)
        self.shared_experts = nn.ModuleList(
            nn.Sequential(
                nn.Linear(h_model, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, h_model),
            )
            for _ in range(num_shared)
        )

        # Routed experts (Mixture-of-Experts)
        self.experts = nn.ModuleList(
            nn.Sequential(
                nn.Linear(h_model, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, h_model),
            )
            for _ in range(num_experts)
        )

        # Router produces gating logits over experts
        self.router = nn.Linear(h_model, num_experts)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [batch_size, h_model]
        returns: [batch_size, h_model]
        """
        # ----- Shared experts -----
        if self.num_shared > 0:
            shared_out = sum(expert(x) for expert in self.shared_experts)
        else:
            shared_out = torch.zeros_like(x)

        # ----- Router & gating -----
        # scores: [B, E]
        logits = self.router(x)
        scores = torch.softmax(logits, dim=-1)

        # top_indices: [B, K], top_scores: [B, K]
        top_scores, top_indices = torch.topk(scores, k=self.top_k, dim=-1)

        # ----- Compute all expert outputs once -----
        # expert_outputs: [B, E, H]
        expert_outputs = torch.stack(
            [expert(x) for expert in self.experts], dim=1
        )

        # ----- Gather top-k experts per token -----
        # top_expert_outputs: [B, K, H]
        top_expert_outputs = expert_outputs.gather(
            dim=1,
            index=top_indices.unsqueeze(-1).expand(-1, -1, self.h_model),
        )

        # ----- Weighted sum of top-k experts -----
        # top_scores: [B, K] -> [B, K, 1]
        weighted_expert_out = (top_expert_outputs * top_scores.unsqueeze(-1)).sum(dim=1)

        # Combine shared + routed experts
        return shared_out + weighted_expert_out


In [5]:
torch.manual_seed(0)

# Mock some data
batch_size = 4
h_model = 16
hidden_dim = 32
num_experts = 4
num_shared = 2
top_k = 2

x = torch.randn(batch_size, h_model)  # [B, H]

moe = DeepSeekMOE(
    h_model=h_model,
    num_experts=num_experts,
    num_shared=num_shared,
    top_k=top_k,
    hidden_dim=hidden_dim,
)

out = moe(x)

print("Input shape :", x.shape)   # torch.Size([4, 16])
print("Output shape:", out.shape) # torch.Size([4, 16])
print("First row (5 dims):", out[0, :5])

Input shape : torch.Size([4, 16])
Output shape: torch.Size([4, 16])
First row (5 dims): tensor([ 0.1381, -0.6746, -0.3216, -0.3531, -0.7540], grad_fn=<SliceBackward0>)
